# test_default_builder.ipynb：逐行演算 `envs/observation/default_builder.py`

这个 notebook 用一份很小的 fake env，逐行说明 `DefaultObservationBuilder` 如何把环境状态组装成 DRL 使用的 observation。

学习顺序：

1. 先看 schema/layout，知道最终输出有哪些 key 和 shape。
2. 再看时间、日历、序列窗口、电价派生特征这些小函数如何计算。
3. 最后用一个 2-agent、4-step 的完整例子，把 `build()` 和 `build_raw()` 的全过程演算出来。

带 `_` 的函数/方法是内部实现细节；这里直接调用它们只是为了学习和验证。


## 0. 准备环境

Notebook 可能从不同工作目录执行，下面先自动找到项目根目录并加入 `sys.path`，避免出现 `No module named 'envs'`。


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "envs").is_dir() and (candidate / "tests").is_dir():
            return candidate
    raise RuntimeError(f"Cannot locate project root from {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from envs.observation.default_builder import (
    DefaultObservationBuilder,
    _calendar_features,
    _pad_sequence,
    _rank_sequence,
    _relative_price_sequence,
    _spread_price_sequence,
    _time_features,
)

np.set_printoptions(precision=4, suppress=True)


def as_value(value):
    array = np.asarray(value)
    if isinstance(value, (pd.DataFrame, pd.Series)):
        return value
    if array.ndim > 0:
        return np.round(array.astype(float), 4).tolist()
    if isinstance(value, float):
        return round(value, 6)
    return value


def show_steps(title: str, input_text: str, rows, final_output=None):
    print()
    print(title)
    print(f"输入：{input_text}")
    display(pd.DataFrame(rows, columns=["步骤", "对应源码/表达式", "本例输入", "中间输出", "解释"]))
    if final_output is not None:
        print("最终输出：")
        display(final_output)


rows = [
    (1, "PROJECT_ROOT = find_project_root(Path.cwd())", str(Path.cwd().resolve()), str(PROJECT_ROOT), "定位仓库根目录。"),
    (2, "sys.path.insert(0, str(PROJECT_ROOT))", "项目根目录", str(PROJECT_ROOT), "让 notebook 能直接 import envs.observation.default_builder。"),
    (3, "import DefaultObservationBuilder 和内部演算函数", "default_builder.py", "导入成功", "后续所有例子都会调用真实代码，而不是重新实现一份逻辑。"),
]
show_steps("准备环境", "当前 notebook 执行目录", rows)



准备环境
输入：当前 notebook 执行目录


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,PROJECT_ROOT = find_project_root(Path.cwd()),D:\GithubProject\MADRL_ESS\tests\testnotebook,D:\GithubProject\MADRL_ESS,定位仓库根目录。
1,2,"sys.path.insert(0, str(PROJECT_ROOT))",项目根目录,D:\GithubProject\MADRL_ESS,让 notebook 能直接 import envs.observation.default...
2,3,import DefaultObservationBuilder 和内部演算函数,default_builder.py,导入成功,后续所有例子都会调用真实代码，而不是重新实现一份逻辑。


## 1. `get_schema()`、`get_layout()`、`zeros()`

作用：先确定 observation 的合同。模型和 replay buffer 依赖这些固定 shape。

本例使用：

- `n_agents = 2`
- `future_horizon = 2`，所以 `sequence_length = future_horizon + 1 = 3`
- local 特征包含全部常见本地特征：`time`、`calendar_time`、`wholesale_price`、`load`、`pv`、`soc`
- sequence 特征包含电价派生序列和 per-agent 的 load/pv 序列


In [2]:
builder = DefaultObservationBuilder(
    local_features=["time", "calendar_time", "wholesale_price", "load", "pv", "soc"],
    sequence_features=[
        "wholesale_price_rank",
        "wholesale_price_relative",
        "wholesale_price_spread",
        "load",
        "pv",
    ],
    future_horizon=2,
    price_spread_scale_eur_per_kwh=0.20,
)
n_agents = 2
schema = builder.get_schema(n_agents)
layout = builder.get_layout(n_agents)
zeros = builder.zeros(n_agents)

rows = [
    (1, "self.sequence_length = future_horizon + 1", "future_horizon=2", builder.sequence_length, "当前步也算在窗口里，所以长度是 3。"),
    (2, "self.local_dim = sum(_LOCAL_DIMS[name] ...)", "time=2, calendar_time=4, wholesale/load/pv/soc 各 1", builder.local_dim, "local 每个 agent 一共有 10 个数。"),
    (3, "schema['local'] = (n_agents, local_dim)", "n_agents=2, local_dim=10", schema["local"], "每个 agent 一行 local 特征。"),
    (4, "schema['safety_local'] = (n_agents, 5)", "soc/load/pv/capacity/p_max", schema["safety_local"], "安全投影器需要原始物理量。"),
    (5, "shared sequence -> (sequence_length,)", "wholesale_price_relative", schema["wholesale_price_relative_seq"], "共享电价不是每个 agent 一份。"),
    (6, "per_agent sequence -> (n_agents, sequence_length)", "load", schema["load_seq"], "负荷和 PV 是每个 agent 各有一条未来窗口。"),
    (7, "zeros(n_agents)", "schema 的每个 shape", {key: value.shape for key, value in zeros.items()}, "按合同生成全零 observation，占位或 reset 前调试可用。"),
    (8, "layout", "schema + feature name", layout["local"], "layout 给训练代码和调试代码说明每个数组的语义。"),
]
show_steps("observation 合同", "2 个 agent，未来窗口 2 步", rows, pd.DataFrame({"key": list(schema), "shape": list(schema.values())}))



observation 合同
输入：2 个 agent，未来窗口 2 步


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,self.sequence_length = future_horizon + 1,future_horizon=2,3,当前步也算在窗口里，所以长度是 3。
1,2,self.local_dim = sum(_LOCAL_DIMS[name] ...),"time=2, calendar_time=4, wholesale/load/pv/soc...",10,local 每个 agent 一共有 10 个数。
2,3,"schema['local'] = (n_agents, local_dim)","n_agents=2, local_dim=10","(2, 10)",每个 agent 一行 local 特征。
3,4,"schema['safety_local'] = (n_agents, 5)",soc/load/pv/capacity/p_max,"(2, 5)",安全投影器需要原始物理量。
4,5,"shared sequence -> (sequence_length,)",wholesale_price_relative,"(3,)",共享电价不是每个 agent 一份。
5,6,"per_agent sequence -> (n_agents, sequence_length)",load,"(2, 3)",负荷和 PV 是每个 agent 各有一条未来窗口。
6,7,zeros(n_agents),schema 的每个 shape,"{'local': (2, 10), 'safety_local': (2, 5), 'wh...",按合同生成全零 observation，占位或 reset 前调试可用。
7,8,layout,schema + feature name,"{'group': 'local', 'scope': 'per_agent', 'dim'...",layout 给训练代码和调试代码说明每个数组的语义。


最终输出：


,key,shape
0,local,"(2, 10)"
1,safety_local,"(2, 5)"
2,wholesale_price_rank_seq,"(3,)"
3,wholesale_price_relative_seq,"(3,)"
4,wholesale_price_spread_seq,"(3,)"
5,load_seq,"(2, 3)"
6,pv_seq,"(2, 3)"


## 2. `_time_features(cur_step, episode_length, n_agents)`

作用：把 episode 内的位置编码成周期时间特征。这个特征只说明“现在在 episode 的哪个相位”，不依赖真实年月日。


In [3]:
cur_step = 1
episode_length = 4
n_agents = 2
phase = 2.0 * np.pi * float(cur_step) / float(episode_length)
features = np.asarray([np.sin(phase), np.cos(phase)], dtype=np.float32)
actual = _time_features(cur_step, episode_length, n_agents)

rows = [
    (1, "phase = 2*pi*cur_step/episode_length", "cur_step=1, episode_length=4", round(float(phase), 6), "第 1 步位于 4 步 episode 的四分之一处。"),
    (2, "features = [sin(phase), cos(phase)]", "phase=pi/2", as_value(features), "sin(pi/2)=1，cos(pi/2)=0。"),
    (3, "features.reshape(1, -1)", "[1, 0]", as_value(features.reshape(1, -1)), "先变成 1 行 2 列。"),
    (4, "np.repeat(..., n_agents, axis=0)", "n_agents=2", as_value(actual), "两个 agent 共享同一个 episode 相位。"),
]
show_steps("_time_features 的逐步演算", "cur_step=1, episode_length=4, n_agents=2", rows, actual)



_time_features 的逐步演算
输入：cur_step=1, episode_length=4, n_agents=2


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,phase = 2*pi*cur_step/episode_length,"cur_step=1, episode_length=4",1.570796,第 1 步位于 4 步 episode 的四分之一处。
1,2,"features = [sin(phase), cos(phase)]",phase=pi/2,"[1.0, 0.0]",sin(pi/2)=1，cos(pi/2)=0。
2,3,"features.reshape(1, -1)","[1, 0]","[[1.0, 0.0]]",先变成 1 行 2 列。
3,4,"np.repeat(..., n_agents, axis=0)",n_agents=2,"[[1.0, 0.0], [1.0, 0.0]]",两个 agent 共享同一个 episode 相位。


最终输出：


array([[1., 0.],
       [1., 0.]], dtype=float32)

## 3. `_calendar_features(timestamp_value, n_agents)`

作用：把真实时间戳编码成日周期和年周期特征。它回答的是“现在是一天中的什么时候、一年中的什么时候”。


In [4]:
timestamp_value = "2020-01-01 06:00:00+00:00"
n_agents = 2
timestamp = pd.Timestamp(timestamp_value)
hour_of_day = float(timestamp.hour) + float(timestamp.minute) / 60.0
day_of_year = float(timestamp.dayofyear - 1) + hour_of_day / 24.0
phases = np.asarray(
    [
        np.sin(2.0 * np.pi * hour_of_day / 24.0),
        np.cos(2.0 * np.pi * hour_of_day / 24.0),
        np.sin(2.0 * np.pi * day_of_year / 365.25),
        np.cos(2.0 * np.pi * day_of_year / 365.25),
    ],
    dtype=np.float32,
)
actual = _calendar_features(timestamp_value, n_agents)

rows = [
    (1, "timestamp = pd.Timestamp(timestamp_value)", timestamp_value, str(timestamp), "字符串先转成 pandas 时间戳。"),
    (2, "hour_of_day = hour + minute/60", "06:00", hour_of_day, "早上 6 点是一日中的第 6 小时。"),
    (3, "day_of_year = dayofyear-1 + hour/24", "1 月 1 日 06:00", round(day_of_year, 6), "从 0 开始数，6 点相当于一年开始后的 0.25 天。"),
    (4, "日周期 sin/cos", "2*pi*6/24", as_value(phases[:2]), "6 点对应日周期的四分之一，相当于 [1, 0]。"),
    (5, "年周期 sin/cos", "2*pi*0.25/365.25", as_value(phases[2:]), "年周期刚开始，所以 sin 很小、cos 接近 1。"),
    (6, "np.repeat(..., n_agents, axis=0)", "n_agents=2", as_value(actual), "真实时间对所有 agent 相同，因此复制两行。"),
]
show_steps("_calendar_features 的逐步演算", timestamp_value, rows, actual)



_calendar_features 的逐步演算
输入：2020-01-01 06:00:00+00:00


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,timestamp = pd.Timestamp(timestamp_value),2020-01-01 06:00:00+00:00,2020-01-01 06:00:00+00:00,字符串先转成 pandas 时间戳。
1,2,hour_of_day = hour + minute/60,06:00,6.0,早上 6 点是一日中的第 6 小时。
2,3,day_of_year = dayofyear-1 + hour/24,1 月 1 日 06:00,0.25,从 0 开始数，6 点相当于一年开始后的 0.25 天。
3,4,日周期 sin/cos,2*pi*6/24,"[1.0, 0.0]","6 点对应日周期的四分之一，相当于 [1, 0]。"
4,5,年周期 sin/cos,2*pi*0.25/365.25,"[0.0043, 1.0]",年周期刚开始，所以 sin 很小、cos 接近 1。
5,6,"np.repeat(..., n_agents, axis=0)",n_agents=2,"[[1.0, 0.0, 0.0043, 1.0], [1.0, 0.0, 0.0043, 1...",真实时间对所有 agent 相同，因此复制两行。


最终输出：


array([[1.    , 0.    , 0.0043, 1.    ],
       [1.    , 0.    , 0.0043, 1.    ]], dtype=float32)

## 4. `_pad_sequence(values, start, length)`

作用：从当前步开始截取未来窗口；如果 episode 尾部不够长，就在后面补 0。共享序列保持一维；per-agent 序列会转置成 `(n_agents, sequence_length)`。


In [5]:
shared_price = np.asarray([0.10, 0.20, 0.50, 0.30], dtype=np.float32)
per_agent_load = np.asarray(
    [
        [10.0, 20.0],
        [11.0, 21.0],
        [12.0, 22.0],
        [13.0, 23.0],
    ],
    dtype=np.float32,
)
start = 1
length = 3
shared_tail = shared_price[start : start + length]
shared_actual = _pad_sequence(shared_price, start, length)
load_tail = per_agent_load[start : start + length]
load_actual = _pad_sequence(per_agent_load, start, length)
end_actual = _pad_sequence(per_agent_load, start=3, length=3)

rows = [
    (1, "array = np.asarray(values, dtype=np.float32)", "shared_price", as_value(shared_price), "先统一成 float32 数组。"),
    (2, "tail = array[start:start+length]", "start=1, length=3", as_value(shared_tail), "共享电价从第 1 步取到第 3 步。"),
    (3, "if array.ndim == 1: return tail", "shared_price 是一维", as_value(shared_actual), "共享信号直接返回 shape=(3,)。"),
    (4, "tail = array[start:start+length]", "per_agent_load", as_value(load_tail), "per-agent 信号先得到 shape=(3, 2)，每行是一个时间步。"),
    (5, "return tail.T", "shape=(3,2)", as_value(load_actual), "转置后变成 shape=(2,3)，每行是一个 agent 的未来窗口。"),
    (6, "pad_rows > 0 时 np.pad(...)", "start=3, length=3", as_value(end_actual), "尾部只剩 1 行，所以每个 agent 后面补两个 0。"),
]
show_steps("_pad_sequence 的逐步演算", "共享电价 + 两个 agent 的负荷矩阵", rows)



_pad_sequence 的逐步演算
输入：共享电价 + 两个 agent 的负荷矩阵


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,"array = np.asarray(values, dtype=np.float32)",shared_price,"[0.1, 0.2, 0.5, 0.3]",先统一成 float32 数组。
1,2,tail = array[start:start+length],"start=1, length=3","[0.2, 0.5, 0.3]",共享电价从第 1 步取到第 3 步。
2,3,if array.ndim == 1: return tail,shared_price 是一维,"[0.2, 0.5, 0.3]","共享信号直接返回 shape=(3,)。"
3,4,tail = array[start:start+length],per_agent_load,"[[11.0, 21.0], [12.0, 22.0], [13.0, 23.0]]","per-agent 信号先得到 shape=(3, 2)，每行是一个时间步。"
4,5,return tail.T,"shape=(3,2)","[[11.0, 12.0, 13.0], [21.0, 22.0, 23.0]]","转置后变成 shape=(2,3)，每行是一个 agent 的未来窗口。"
5,6,pad_rows > 0 时 np.pad(...),"start=3, length=3","[[13.0, 0.0, 0.0], [23.0, 0.0, 0.0]]",尾部只剩 1 行，所以每个 agent 后面补两个 0。


## 5. 电价派生序列：rank、relative、spread

这三个特征都从同一个 `wholesale_price` 未来窗口派生：

- `rank`：当前窗口内的相对排序，范围 0 到 1。
- `relative`：按当前窗口的最高最低价缩放到 -1 到 1。
- `spread`：当前窗口价差强度，范围 0 到 1，并复制成同样长度。


In [6]:
price_window = np.asarray([0.20, 0.50, 0.30], dtype=np.float32)
order = np.argsort(price_window, kind="mergesort")
manual_ranks = np.empty((price_window.size,), dtype=np.float32)
manual_ranks[order] = np.arange(price_window.size, dtype=np.float32)
rank_actual = _rank_sequence(price_window)
min_value = float(np.min(price_window))
max_value = float(np.max(price_window))
spread = max_value - min_value
relative_actual = _relative_price_sequence(price_window)
spread_actual = _spread_price_sequence(price_window, scale_eur_per_kwh=0.20)

rows = [
    (1, "array = values.reshape(-1)", "[0.20, 0.50, 0.30]", as_value(price_window), "未来 3 步共享电价窗口。"),
    (2, "order = np.argsort(array)", "从低到高排序", as_value(order), "0.20 最低在位置 0，0.30 次低在位置 2，0.50 最高在位置 1。"),
    (3, "ranks[order] = arange(length)", "order=[0,2,1]", as_value(manual_ranks), "原位置 0/1/2 的名次分别是 0/2/1。"),
    (4, "rank = ranks/(length-1)", "length=3", as_value(rank_actual), "除以 2 后得到 [0, 1, 0.5]。"),
    (5, "min/max/spread", "窗口内最高最低价", {"min": min_value, "max": max_value, "spread": round(spread, 4)}, "价差是 0.50-0.20=0.30。"),
    (6, "relative = 2*(array-min)/spread - 1", "spread=0.30", as_value(relative_actual), "最低价变 -1，最高价变 1，中间价约 -0.3333。"),
    (7, "spread_value = clip(spread/scale, 0, 1)", "spread=0.30, scale=0.20", as_value(spread_actual), "0.30/0.20=1.5，被裁剪成 1，然后复制成长度 3。"),
]
show_steps("电价派生序列的逐步演算", "price_window=[0.20, 0.50, 0.30]", rows)



电价派生序列的逐步演算
输入：price_window=[0.20, 0.50, 0.30]


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,array = values.reshape(-1),"[0.20, 0.50, 0.30]","[0.2, 0.5, 0.3]",未来 3 步共享电价窗口。
1,2,order = np.argsort(array),从低到高排序,"[0.0, 2.0, 1.0]",0.20 最低在位置 0，0.30 次低在位置 2，0.50 最高在位置 1。
2,3,ranks[order] = arange(length),"order=[0,2,1]","[0.0, 2.0, 1.0]",原位置 0/1/2 的名次分别是 0/2/1。
3,4,rank = ranks/(length-1),length=3,"[0.0, 1.0, 0.5]","除以 2 后得到 [0, 1, 0.5]。"
4,5,min/max/spread,窗口内最高最低价,"{'min': 0.20000000298023224, 'max': 0.5, 'spre...",价差是 0.50-0.20=0.30。
5,6,relative = 2*(array-min)/spread - 1,spread=0.30,"[-1.0, 1.0, -0.3333]",最低价变 -1，最高价变 1，中间价约 -0.3333。
6,7,"spread_value = clip(spread/scale, 0, 1)","spread=0.30, scale=0.20","[1.0, 1.0, 1.0]",0.30/0.20=1.5，被裁剪成 1，然后复制成长度 3。


## 6. 构造一个最小 fake env

`DefaultObservationBuilder` 不直接拥有电价、负荷、PV、SOC 数据，它从 env 读取这些字段。下面造一个 2-agent、4-step 的环境对象，数值都可以手算。


In [7]:
class DemoEnv:
    def __init__(self):
        self.n = 2
        self.cur_step = 1
        self.episode_length = 4
        self.soc = np.asarray([0.25, 0.75], dtype=np.float32)
        self.agent_c_bat = np.asarray([10.0, 20.0], dtype=np.float32)
        self.agent_p_max = np.asarray([5.0, 8.0], dtype=np.float32)
        self.signals = {
            "wholesale_price": np.asarray([0.10, 0.20, 0.50, 0.30], dtype=np.float32),
            "load": np.asarray(
                [[10.0, 20.0], [11.0, 21.0], [12.0, 22.0], [13.0, 23.0]],
                dtype=np.float32,
            ),
            "pv": np.asarray(
                [[1.0, 2.0], [2.0, 4.0], [3.0, 6.0], [4.0, 8.0]],
                dtype=np.float32,
            ),
        }
        self.history_signals = {
            "wholesale_price": np.asarray([], dtype=np.float32),
            "load": np.zeros((0, self.n), dtype=np.float32),
            "pv": np.zeros((0, self.n), dtype=np.float32),
        }
        self.history_timestamps = []
        self.episode_meta = {
            "timestamps": [
                "2020-01-01 05:45:00+00:00",
                "2020-01-01 06:00:00+00:00",
                "2020-01-01 06:15:00+00:00",
                "2020-01-01 06:30:00+00:00",
            ]
        }
        self._episode_precomputed = {}
        self.forecaster = None

    def get_signal(self, name: str):
        return self.signals[name]

    def get_signal_step(self, name: str, step=None):
        step = self.cur_step if step is None else int(step)
        value = self.signals[name][step]
        array = np.asarray(value, dtype=np.float32)
        if array.ndim == 0:
            return float(array)
        return array

    def _current_safety_local(self):
        return np.column_stack(
            [
                self.soc,
                self.get_signal_step("load"),
                self.get_signal_step("pv"),
                self.agent_c_bat,
                self.agent_p_max,
            ]
        ).astype(np.float32)


env = DemoEnv()
current_price = env.get_signal_step("wholesale_price")
current_load = env.get_signal_step("load")
current_pv = env.get_signal_step("pv")
safety_local = env._current_safety_local()

rows = [
    (1, "env.n", "两个 agent", env.n, "observation 第一维都是 agent 数。"),
    (2, "env.cur_step", "当前在第 1 步", env.cur_step, "所有 local 特征读取当前步。"),
    (3, "env.get_signal_step('wholesale_price')", "price[1]", current_price, "共享电价当前值是 0.20。"),
    (4, "env.get_signal_step('load')", "load[1]", as_value(current_load), "当前步两个 agent 的负荷是 11 和 21。"),
    (5, "env.get_signal_step('pv')", "pv[1]", as_value(current_pv), "当前步两个 agent 的 PV 是 2 和 4。"),
    (6, "env._current_safety_local()", "soc/load/pv/capacity/p_max", as_value(safety_local), "安全投影器需要这些未归一化物理量。"),
]
show_steps("fake env 的输入数据", "2-agent, 4-step", rows, pd.DataFrame(safety_local, columns=["soc_raw", "load_raw", "pv_raw", "battery_capacity_kwh", "p_max_kw"]))



fake env 的输入数据
输入：2-agent, 4-step


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,env.n,两个 agent,2,observation 第一维都是 agent 数。
1,2,env.cur_step,当前在第 1 步,1,所有 local 特征读取当前步。
2,3,env.get_signal_step('wholesale_price'),price[1],0.2,共享电价当前值是 0.20。
3,4,env.get_signal_step('load'),load[1],"[11.0, 21.0]",当前步两个 agent 的负荷是 11 和 21。
4,5,env.get_signal_step('pv'),pv[1],"[2.0, 4.0]",当前步两个 agent 的 PV 是 2 和 4。
5,6,env._current_safety_local(),soc/load/pv/capacity/p_max,"[[0.25, 11.0, 2.0, 10.0, 5.0], [0.75, 21.0, 4....",安全投影器需要这些未归一化物理量。


最终输出：


,soc_raw,load_raw,pv_raw,battery_capacity_kwh,p_max_kw
0,0.25,11.0,2.0,10.0,5.0
1,0.75,21.0,4.0,20.0,8.0


## 7. `_local_feature(env, name)`

作用：把当前步的本地特征都变成 `(n_agents, feature_dim)`，最后横向拼成 `obs['local']`。


In [8]:
local_parts = []
local_rows = []
for name in builder.local_feature_names:
    values = builder._local_feature(env, name)
    local_parts.append(values)
    local_rows.append(
        (
            len(local_rows) + 1,
            f"_local_feature(env, '{name}')",
            f"cur_step={env.cur_step}",
            as_value(values),
            "每个 local 特征都返回 per-agent 矩阵，随后按列拼接。",
        )
    )

local = np.concatenate(local_parts, axis=1).astype(np.float32)
local_rows.append(
    (
        len(local_rows) + 1,
        "np.concatenate(local_parts, axis=1)",
        "time/calendar/price/load/pv/soc",
        {"shape": local.shape, "values": as_value(local)},
        "最终 local 每个 agent 一行，共 10 列。",
    )
)
show_steps("_local_feature 的逐步演算", "读取 fake env 当前步", local_rows, pd.DataFrame(local, columns=["time_sin", "time_cos", "day_sin", "day_cos", "year_sin", "year_cos", "price", "load", "pv", "soc"]))



_local_feature 的逐步演算
输入：读取 fake env 当前步


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,"_local_feature(env, 'time')",cur_step=1,"[[1.0, 0.0], [1.0, 0.0]]",每个 local 特征都返回 per-agent 矩阵，随后按列拼接。
1,2,"_local_feature(env, 'calendar_time')",cur_step=1,"[[1.0, 0.0, 0.0043, 1.0], [1.0, 0.0, 0.0043, 1...",每个 local 特征都返回 per-agent 矩阵，随后按列拼接。
2,3,"_local_feature(env, 'wholesale_price')",cur_step=1,"[[0.2], [0.2]]",每个 local 特征都返回 per-agent 矩阵，随后按列拼接。
3,4,"_local_feature(env, 'load')",cur_step=1,"[[11.0], [21.0]]",每个 local 特征都返回 per-agent 矩阵，随后按列拼接。
4,5,"_local_feature(env, 'pv')",cur_step=1,"[[2.0], [4.0]]",每个 local 特征都返回 per-agent 矩阵，随后按列拼接。
5,6,"_local_feature(env, 'soc')",cur_step=1,"[[0.25], [0.75]]",每个 local 特征都返回 per-agent 矩阵，随后按列拼接。
6,7,"np.concatenate(local_parts, axis=1)",time/calendar/price/load/pv/soc,"{'shape': (2, 10), 'values': [[1.0, 0.0, 1.0, ...",最终 local 每个 agent 一行，共 10 列。


最终输出：


,time_sin,time_cos,day_sin,day_cos,year_sin,year_cos,price,load,pv,soc
0,1.0,6.123234e-17,1.0,6.123234e-17,0.004301,0.999991,0.2,11.0,2.0,0.25
1,1.0,6.123234e-17,1.0,6.123234e-17,0.004301,0.999991,0.2,21.0,4.0,0.75


## 8. `_sequence_feature(env, name)`

作用：为每个序列特征取当前步开始的未来窗口。没有 forecaster 时，直接从当前 episode 的已知信号中截取；电价派生特征会先取 `wholesale_price`，再做 rank/relative/spread。


In [9]:
sequence_cache = {}
sequence_rows = []
sequence_outputs = {}
for name in builder.sequence_feature_names:
    values = builder._sequence_feature(env, name, sequence_cache)
    sequence_outputs[f"{name}_seq"] = values
    sequence_rows.append(
        (
            len(sequence_rows) + 1,
            f"_sequence_feature(env, '{name}', cache)",
            f"cur_step={env.cur_step}, sequence_length={builder.sequence_length}",
            {"shape": values.shape, "values": as_value(values)},
            "shared 电价序列是一维；load/pv per-agent 序列是二维。",
        )
    )

sequence_rows.append(
    (
        len(sequence_rows) + 1,
        "sequence_cache",
        "rank/relative/spread 都依赖 wholesale_price",
        {key: as_value(value) for key, value in sequence_cache.items()},
        "同一轮 build 中，基础 wholesale_price 序列会被缓存并复用。",
    )
)
show_steps("_sequence_feature 的逐步演算", "从当前步截取未来 3 个点", sequence_rows, pd.DataFrame({key: [as_value(value)] for key, value in sequence_outputs.items()}))



_sequence_feature 的逐步演算
输入：从当前步截取未来 3 个点


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,"_sequence_feature(env, 'wholesale_price_rank',...","cur_step=1, sequence_length=3","{'shape': (3,), 'values': [0.0, 1.0, 0.5]}",shared 电价序列是一维；load/pv per-agent 序列是二维。
1,2,"_sequence_feature(env, 'wholesale_price_relati...","cur_step=1, sequence_length=3","{'shape': (3,), 'values': [-1.0, 1.0, -0.3333]}",shared 电价序列是一维；load/pv per-agent 序列是二维。
2,3,"_sequence_feature(env, 'wholesale_price_spread...","cur_step=1, sequence_length=3","{'shape': (3,), 'values': [1.0, 1.0, 1.0]}",shared 电价序列是一维；load/pv per-agent 序列是二维。
3,4,"_sequence_feature(env, 'load', cache)","cur_step=1, sequence_length=3","{'shape': (2, 3), 'values': [[11.0, 12.0, 13.0...",shared 电价序列是一维；load/pv per-agent 序列是二维。
4,5,"_sequence_feature(env, 'pv', cache)","cur_step=1, sequence_length=3","{'shape': (2, 3), 'values': [[2.0, 3.0, 4.0], ...",shared 电价序列是一维；load/pv per-agent 序列是二维。
5,6,sequence_cache,rank/relative/spread 都依赖 wholesale_price,"{'wholesale_price': [0.2, 0.5, 0.3], 'wholesal...",同一轮 build 中，基础 wholesale_price 序列会被缓存并复用。


最终输出：


,wholesale_price_rank_seq,wholesale_price_relative_seq,wholesale_price_spread_seq,load_seq,pv_seq
0,"[0.0, 1.0, 0.5]","[-1.0, 1.0, -0.3333]","[1.0, 1.0, 1.0]","[[11.0, 12.0, 13.0], [21.0, 22.0, 23.0]]","[[2.0, 3.0, 4.0], [4.0, 6.0, 8.0]]"


## 9. `build(env)` 的完整输出

作用：训练和推理主线调用 `build()`，得到模型要吃的 observation 字典。这里没有 normalizer，所以输出就是原始计算值。


In [10]:
obs = builder.build(env)
obs_summary = pd.DataFrame(
    [
        {"key": key, "shape": value.shape, "values": as_value(value)}
        for key, value in obs.items()
    ]
)

rows = [
    (1, "local_parts = [_normalize(... _local_feature ...)]", "normalize=True 但 normalizer=None", obs["local"].shape, "没有 normalizer 时，_normalize 只是转成 float32。"),
    (2, "obs['safety_local'] = env._current_safety_local()", "fake env 当前安全物理量", as_value(obs["safety_local"]), "safety_local 始终保留原始物理量，不从 local 拼出来。"),
    (3, "for name in sequence_feature_names", builder.sequence_feature_names, list(obs.keys()), "每个 sequence 特征都会以 '<name>_seq' 形式进入 observation。"),
    (4, "return obs", "local + safety_local + sequence", {key: value.shape for key, value in obs.items()}, "这就是 DRL policy/replay buffer 看到的 observation 合同。"),
]
show_steps("build(env) 的完整演算", "fake env + builder", rows, obs_summary)



build(env) 的完整演算
输入：fake env + builder


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,local_parts = [_normalize(... _local_feature ....,normalize=True 但 normalizer=None,"(2, 10)",没有 normalizer 时，_normalize 只是转成 float32。
1,2,obs['safety_local'] = env._current_safety_local(),fake env 当前安全物理量,"[[0.25, 11.0, 2.0, 10.0, 5.0], [0.75, 21.0, 4....",safety_local 始终保留原始物理量，不从 local 拼出来。
2,3,for name in sequence_feature_names,"[wholesale_price_rank, wholesale_price_relativ...","[local, safety_local, wholesale_price_rank_seq...",每个 sequence 特征都会以 '<name>_seq' 形式进入 observation。
3,4,return obs,local + safety_local + sequence,"{'local': (2, 10), 'safety_local': (2, 5), 'wh...",这就是 DRL policy/replay buffer 看到的 observation 合同。


最终输出：


,key,shape,values
0,local,"(2, 10)","[[1.0, 0.0, 1.0, 0.0, 0.0043, 1.0, 0.2, 11.0, ..."
1,safety_local,"(2, 5)","[[0.25, 11.0, 2.0, 10.0, 5.0], [0.75, 21.0, 4...."
2,wholesale_price_rank_seq,"(3,)","[0.0, 1.0, 0.5]"
3,wholesale_price_relative_seq,"(3,)","[-1.0, 1.0, -0.3333]"
4,wholesale_price_spread_seq,"(3,)","[1.0, 1.0, 1.0]"
5,load_seq,"(2, 3)","[[11.0, 12.0, 13.0], [21.0, 22.0, 23.0]]"
6,pv_seq,"(2, 3)","[[2.0, 3.0, 4.0], [4.0, 6.0, 8.0]]"


## 10. `build_raw(env)` 为什么会额外补 `wholesale_price_seq`

作用：如果配置里只用了 `wholesale_price_relative`、`wholesale_price_spread` 这类派生特征，raw observation 仍然需要补上原始 `wholesale_price_seq`，方便后续做统计、调试或 normalizer 拟合。


In [11]:
raw_obs = builder.build_raw(env)
raw_keys = list(raw_obs.keys())
raw_sequence_names = builder._raw_sequence_feature_names()
raw_summary = pd.DataFrame(
    [
        {"key": key, "shape": value.shape, "values": as_value(value)}
        for key, value in raw_obs.items()
    ]
)

rows = [
    (1, "names = list(self.sequence_feature_names)", builder.sequence_feature_names, builder.sequence_feature_names, "先拿配置中真正给模型使用的序列特征。"),
    (2, "dependency = 'wholesale_price'", "rank/relative/spread 的依赖", raw_sequence_names, "派生电价特征依赖原始电价，所以 raw 会把 wholesale_price 插到前面。"),
    (3, "build_raw(... normalize=False ...)", "raw_sequence_feature_names", raw_keys, "raw 输出包含 wholesale_price_seq，但 build 输出不一定包含它。"),
    (4, "wholesale_price_seq", "cur_step=1", as_value(raw_obs["wholesale_price_seq"]), "这是派生特征的原始输入窗口：[0.20, 0.50, 0.30]。"),
]
show_steps("build_raw(env) 的逐步演算", "派生电价序列需要原始依赖", rows, raw_summary)



build_raw(env) 的逐步演算
输入：派生电价序列需要原始依赖


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,names = list(self.sequence_feature_names),"[wholesale_price_rank, wholesale_price_relativ...","[wholesale_price_rank, wholesale_price_relativ...",先拿配置中真正给模型使用的序列特征。
1,2,dependency = 'wholesale_price',rank/relative/spread 的依赖,"[wholesale_price, wholesale_price_rank, wholes...",派生电价特征依赖原始电价，所以 raw 会把 wholesale_price 插到前面。
2,3,build_raw(... normalize=False ...),raw_sequence_feature_names,"[local, safety_local, wholesale_price_seq, who...",raw 输出包含 wholesale_price_seq，但 build 输出不一定包含它。
3,4,wholesale_price_seq,cur_step=1,"[0.2, 0.5, 0.3]","这是派生特征的原始输入窗口：[0.20, 0.50, 0.30]。"


最终输出：


,key,shape,values
0,local,"(2, 10)","[[1.0, 0.0, 1.0, 0.0, 0.0043, 1.0, 0.2, 11.0, ..."
1,safety_local,"(2, 5)","[[0.25, 11.0, 2.0, 10.0, 5.0], [0.75, 21.0, 4...."
2,wholesale_price_seq,"(3,)","[0.2, 0.5, 0.3]"
3,wholesale_price_rank_seq,"(3,)","[0.0, 1.0, 0.5]"
4,wholesale_price_relative_seq,"(3,)","[-1.0, 1.0, -0.3333]"
5,wholesale_price_spread_seq,"(3,)","[1.0, 1.0, 1.0]"
6,load_seq,"(2, 3)","[[11.0, 12.0, 13.0], [21.0, 22.0, 23.0]]"
7,pv_seq,"(2, 3)","[[2.0, 3.0, 4.0], [4.0, 6.0, 8.0]]"


## 11. normalizer 分支

作用：`build()` 会在有 normalizer 时归一化 local/sequence 特征；`build_raw()` 明确用于原始统计和调试，所以不归一化。


In [12]:
class DivideByTenNormalizer:
    def transform_local(self, feature_name: str, values):
        return np.asarray(values, dtype=np.float32) / 10.0

    def transform_sequence(self, feature_name: str, values):
        return np.asarray(values, dtype=np.float32) / 10.0


norm_builder = DefaultObservationBuilder(
    local_features=["load"],
    sequence_features=["load"],
    future_horizon=2,
    normalizer=DivideByTenNormalizer(),
)
raw_load_obs = norm_builder.build_raw(env)
normalized_load_obs = norm_builder.build(env)

rows = [
    (1, "build_raw(env)", "normalize=False", as_value(raw_load_obs["local"]), "raw local load 保持 [11, 21]。"),
    (2, "build(env)", "normalize=True", as_value(normalized_load_obs["local"]), "normalizer.transform_local 把 local load 除以 10。"),
    (3, "build_raw(env)['load_seq']", "未来负荷窗口", as_value(raw_load_obs["load_seq"]), "raw sequence 保持原始 kW 示例值。"),
    (4, "build(env)['load_seq']", "未来负荷窗口", as_value(normalized_load_obs["load_seq"]), "normalizer.transform_sequence 把 sequence load 除以 10。"),
    (5, "safety_local", "raw vs build", as_value(normalized_load_obs["safety_local"]), "safety_local 不经过 normalizer，仍是安全投影器需要的物理量。"),
]
show_steps("normalizer 分支的逐步演算", "一个把数值除以 10 的演示 normalizer", rows)



normalizer 分支的逐步演算
输入：一个把数值除以 10 的演示 normalizer


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,build_raw(env),normalize=False,"[[11.0], [21.0]]","raw local load 保持 [11, 21]。"
1,2,build(env),normalize=True,"[[1.1], [2.1]]",normalizer.transform_local 把 local load 除以 10。
2,3,build_raw(env)['load_seq'],未来负荷窗口,"[[11.0, 12.0, 13.0], [21.0, 22.0, 23.0]]",raw sequence 保持原始 kW 示例值。
3,4,build(env)['load_seq'],未来负荷窗口,"[[1.1, 1.2, 1.3], [2.1, 2.2, 2.3]]",normalizer.transform_sequence 把 sequence load ...
4,5,safety_local,raw vs build,"[[0.25, 11.0, 2.0, 10.0, 5.0], [0.75, 21.0, 4....",safety_local 不经过 normalizer，仍是安全投影器需要的物理量。


## 12. forecaster 分支和 sequence cache

作用：如果 env 有 forecaster，普通序列不再直接从 episode 信号截取，而是用历史序列预测未来窗口。派生电价特征共用同一个基础 `wholesale_price` 预测结果，避免重复调用 forecaster。


In [13]:
class CountingForecaster:
    def __init__(self):
        self.calls = []

    def predict(self, history, length: int, signal_name: str, history_timestamps):
        self.calls.append(
            {
                "signal_name": signal_name,
                "history": np.asarray(history, dtype=np.float32).copy(),
                "length": int(length),
                "history_timestamps": list(history_timestamps),
            }
        )
        if signal_name == "wholesale_price":
            return np.asarray([0.20, 0.50, 0.30], dtype=np.float32)
        raise AssertionError(f"Unexpected signal_name={signal_name}")


forecast_env = DemoEnv()
forecast_env.history_signals["wholesale_price"] = np.asarray([0.05], dtype=np.float32)
forecast_env.history_timestamps = ["2020-01-01 05:30:00+00:00"]
forecast_env.forecaster = CountingForecaster()
forecast_builder = DefaultObservationBuilder(
    local_features=[],
    sequence_features=["wholesale_price_relative", "wholesale_price_spread"],
    future_horizon=2,
)
forecast_obs = forecast_builder.build(forecast_env)
call = forecast_env.forecaster.calls[0]

rows = [
    (1, "_signal_history(env, 'wholesale_price')", "history prefix + 当前 episode 到 cur_step", as_value(call["history"]), "历史 [0.05] 加上当前 episode 的 [0.10, 0.20]。"),
    (2, "history_timestamps", "历史时间 + episode 时间到 cur_step", call["history_timestamps"], "forecaster 可以知道每个历史点对应的时间。"),
    (3, "forecaster.predict(..., length=3)", "signal_name='wholesale_price'", as_value([0.20, 0.50, 0.30]), "演示 forecaster 返回未来 3 步电价。"),
    (4, "wholesale_price_relative", "复用预测电价", as_value(forecast_obs["wholesale_price_relative_seq"]), "预测电价被缩放到 -1 到 1。"),
    (5, "wholesale_price_spread", "复用预测电价", as_value(forecast_obs["wholesale_price_spread_seq"]), "同一个预测电价序列再次用于价差强度。"),
    (6, "len(forecaster.calls)", "两个派生特征", len(forecast_env.forecaster.calls), "因为 sequence_cache 存在，基础 wholesale_price 只预测了一次。"),
]
show_steps("forecaster + cache 的逐步演算", "relative 和 spread 共用 wholesale_price 预测", rows)



forecaster + cache 的逐步演算
输入：relative 和 spread 共用 wholesale_price 预测


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,"_signal_history(env, 'wholesale_price')",history prefix + 当前 episode 到 cur_step,"[0.05, 0.1, 0.2]","历史 [0.05] 加上当前 episode 的 [0.10, 0.20]。"
1,2,history_timestamps,历史时间 + episode 时间到 cur_step,"[2020-01-01 05:30:00+00:00, 2020-01-01 05:45:0...",forecaster 可以知道每个历史点对应的时间。
2,3,"forecaster.predict(..., length=3)",signal_name='wholesale_price',"[0.2, 0.5, 0.3]",演示 forecaster 返回未来 3 步电价。
3,4,wholesale_price_relative,复用预测电价,"[-1.0, 1.0, -0.3333]",预测电价被缩放到 -1 到 1。
4,5,wholesale_price_spread,复用预测电价,"[1.0, 1.0, 1.0]",同一个预测电价序列再次用于价差强度。
5,6,len(forecaster.calls),两个派生特征,1,因为 sequence_cache 存在，基础 wholesale_price 只预测了一次。


## 13. precomputed 分支的 shape 合同

作用：当 observation 序列已经提前算好时，builder 会直接从 `env._episode_precomputed` 读取，并严格检查 shape。shape 不一致会立即报错，避免旧共享数据悄悄进入训练。


In [14]:
pre_builder = DefaultObservationBuilder(
    local_features=[],
    sequence_features=["load"],
    future_horizon=2,
    precomputed=True,
)
pre_env = DemoEnv()
precomputed_load = np.asarray(
    [
        [[10.0, 11.0, 12.0], [20.0, 21.0, 22.0]],
        [[11.0, 12.0, 13.0], [21.0, 22.0, 23.0]],
        [[12.0, 13.0, 0.0], [22.0, 23.0, 0.0]],
        [[13.0, 0.0, 0.0], [23.0, 0.0, 0.0]],
    ],
    dtype=np.float32,
)
pre_env._episode_precomputed = {"load_seq": precomputed_load}
good = pre_builder._sequence_feature(pre_env, "load", cache={})

bad_env = DemoEnv()
bad_env._precomputed_data_dir = "demo/shared-data"
bad_env._episode_precomputed = {"load_seq": np.zeros((4, 2, 2), dtype=np.float32)}
try:
    pre_builder._sequence_feature(bad_env, "load", cache={})
except ValueError as exc:
    error_text = str(exc).split(". Expected shared-data")[0]

rows = [
    (1, "cache_key = f'{name}_seq'", "name='load'", "load_seq", "precomputed 数据按 '<feature>_seq' 命名。"),
    (2, "values = env._episode_precomputed[cache_key][cur_step]", "cur_step=1", as_value(good), "直接读取第 1 步预计算好的 load 窗口。"),
    (3, "expected_shape = (env.n, sequence_length)", "env.n=2, sequence_length=3", (2, 3), "load 是 per-agent 序列，所以必须是 2 行 3 列。"),
    (4, "tuple(values.shape) != expected_shape", "坏例子 shape=(2,2)", error_text, "shape 不匹配时立即 ValueError，避免训练 silently 使用旧窗口长度。"),
]
show_steps("precomputed shape 合同的逐步演算", "load_seq 预计算缓存", rows)



precomputed shape 合同的逐步演算
输入：load_seq 预计算缓存


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,cache_key = f'{name}_seq',name='load',load_seq,precomputed 数据按 '<feature>_seq' 命名。
1,2,values = env._episode_precomputed[cache_key][c...,cur_step=1,"[[11.0, 12.0, 13.0], [21.0, 22.0, 23.0]]",直接读取第 1 步预计算好的 load 窗口。
2,3,"expected_shape = (env.n, sequence_length)","env.n=2, sequence_length=3","(2, 3)",load 是 per-agent 序列，所以必须是 2 行 3 列。
3,4,tuple(values.shape) != expected_shape,"坏例子 shape=(2,2)",Precomputed observation horizon contract misma...,shape 不匹配时立即 ValueError，避免训练 silently 使用旧窗口长度。


## 14. 总结：`DefaultObservationBuilder` 的输入输出边界

可以把它理解成 observation 装配器：它不负责产生原始数据，也不负责训练模型；它只从 env 当前状态、未来窗口或 forecaster/precomputed 缓存中取数，然后按固定合同组装成字典。


In [15]:
summary_rows = [
    (1, "输入 env", "cur_step/soc/signals/episode_meta/forecaster/precomputed", "环境当前状态", "builder 从 env 读数据，但不拥有数据。"),
    (2, "local", "当前步 per-agent 特征", "shape=(n_agents, local_dim)", "给 actor/critic 当前局部状态。"),
    (3, "safety_local", "soc/load/pv/capacity/p_max 原始量", "shape=(n_agents, 5)", "给安全投影器使用，保持物理单位。"),
    (4, "*_seq", "当前步到未来 horizon 的窗口", "shared: (T,), per-agent: (n_agents,T)", "给模型看未来价格、负荷、PV 等序列信息。"),
    (5, "build_raw()", "normalize=False + 补派生依赖", "额外可能有 wholesale_price_seq", "用于统计拟合和调试。"),
    (6, "build()", "normalize=True", "训练/推理 observation", "这是主线 DRL 看到的输出。"),
]
show_steps("DefaultObservationBuilder 总结", "env -> observation dict", summary_rows)



DefaultObservationBuilder 总结
输入：env -> observation dict


,步骤,对应源码/表达式,本例输入,中间输出,解释
0,1,输入 env,cur_step/soc/signals/episode_meta/forecaster/p...,环境当前状态,builder 从 env 读数据，但不拥有数据。
1,2,local,当前步 per-agent 特征,"shape=(n_agents, local_dim)",给 actor/critic 当前局部状态。
2,3,safety_local,soc/load/pv/capacity/p_max 原始量,"shape=(n_agents, 5)",给安全投影器使用，保持物理单位。
3,4,*_seq,当前步到未来 horizon 的窗口,"shared: (T,), per-agent: (n_agents,T)",给模型看未来价格、负荷、PV 等序列信息。
4,5,build_raw(),normalize=False + 补派生依赖,额外可能有 wholesale_price_seq,用于统计拟合和调试。
5,6,build(),normalize=True,训练/推理 observation,这是主线 DRL 看到的输出。
